# 03 · Ablation grid
Trains every (variant × head mode × training mode) combination with a shorter budget and reports window metrics,
sequential metrics (own-estimate feedback), parameter count and MACs per inference. Use this to decide:
1. does attention earn its ~60 % of the compute? (`patch` vs `noattn`)
2. do deltas + fusion beat the absolute head on ordered data?
3. does integration-in-the-loop (`K=8`) reduce drift?

In [ ]:
# --- locate emg_common.py: Kaggle utility script, attached dataset, or local folder ---
import os, sys, glob, json
for p in ['/kaggle/usr/lib/emg_common', '/kaggle/usr/lib/emg-common', '/kaggle/input/emg-common', '.', '..', '../kaggle_scripts', 'kaggle_scripts']:
    if os.path.exists(os.path.join(p, 'emg_common.py')):
        sys.path.insert(0, os.path.abspath(p)); break
else:
    raise ImportError('emg_common.py not found: add it as a Utility Script (File > Add utility script) or dataset')
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
import emg_common as ec
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
WORK = os.environ.get('EMG_WORK', '/kaggle/working')           # EMG_* env vars exist only for local smoke tests
BASE = os.environ.get('EMG_BASE_PATH', ec.Config.base_path)
EPOCHS_OVERRIDE = int(os.environ['EMG_EPOCHS']) if 'EMG_EPOCHS' in os.environ else None
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), '| emg_common from', ec.__file__)

In [ ]:
# processed data: output of 01_preprocess_split (same session) or attached as a dataset
candidates = [f'{WORK}/db2_processed'] + sorted(glob.glob('/kaggle/input/*/db2_processed')) + sorted(glob.glob('/kaggle/input/*'))
npz_files = []
for c in candidates:
    npz_files = sorted(glob.glob(f'{c}/*/data.npz'))
    if npz_files: break
assert npz_files, 'no processed data found: run 01_preprocess_split first or attach its output'
print(len(npz_files), 'subjects:', [os.path.basename(os.path.dirname(f)) for f in npz_files])

In [ ]:
base = dict(processed_dir=os.path.dirname(os.path.dirname(npz_files[0])), save_dir=f'{WORK}/outputs_grid', epochs=15, patience=5, qat_epochs=0,
            use_wandb=os.environ.get('EMG_WANDB', '1') == '1', project_name='ninapro_db2_ablation')
GRID = [
    dict(variant='roformer', heads='absolute', cond_prev=False, unroll_steps=0),   # current design
    dict(variant='patch',    heads='absolute', cond_prev=False, unroll_steps=0),
    dict(variant='noattn',   heads='absolute', cond_prev=False, unroll_steps=0),
    dict(variant='patch',    heads='both',     cond_prev=True,  unroll_steps=0),
    dict(variant='patch',    heads='both',     cond_prev=True,  unroll_steps=8),
    dict(variant='noattn',   heads='both',     cond_prev=True,  unroll_steps=8),
    dict(variant='causal',   heads='both',     cond_prev=True,  unroll_steps=8),
]
if EPOCHS_OVERRIDE is not None:
    base['epochs'] = EPOCHS_OVERRIDE; GRID = GRID[:int(os.environ.get('EMG_GRID_N', len(GRID)))]
rest_pose, thr = ec.rest_pose_from_train(npz_files), ec.rest_energy_threshold(npz_files)
device = ec.get_device(); rows = []

In [ ]:
for g in GRID:
    cfg = ec.Config(**base, **g).apply_variant()
    cfg.run_name = f"{g['variant']}_{g['heads']}_prev{int(g['cond_prev'])}_K{g['unroll_steps']}"
    run = ec.init_wandb(cfg)
    model, info = ec.train_model(cfg, npz_files, run=run, tag=cfg.run_name)
    _, val_loader = ec.make_loaders(ec.Config.from_dict({**cfg.to_dict(), 'unroll_steps': 0}), npz_files)
    wm = ec.window_metrics(ec.predict_windows(model, val_loader, cfg, device))
    rep = ec.evaluate_sequential(model, npz_files, cfg, device, rest_pose=rest_pose, rest_threshold=thr, feedback=True)
    row = {**g, 'params': ec.count_params(model), 'MACs': ec.estimate_macs(cfg)['total'], 'tokens': cfg.num_tokens,
           'best_val_loss': min(h['val_loss'] for h in info['history']), **{f'win_{k}': v for k, v in wm.items()},
           **{f'seq_{k}_rmse': v['rmse'] for k, v in rep.items()}, **{f'seq_{k}_r2': v['r2'] for k, v in rep.items()}}
    rows.append(row); ec.wandb_log(run, row)
    if run is not None: run.finish()
    pd.DataFrame(rows).to_csv(f"{base['save_dir']}/ablation_results.csv", index=False)
results = pd.DataFrame(rows); display(results)

In [ ]:
seq_cols = [c for c in results.columns if c.startswith('seq_') and c.endswith('_rmse')]
best_seq = results[seq_cols].min(axis=1)
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
labels = results['variant'] + '/' + results['heads'] + '/K' + results['unroll_steps'].astype(str)
ax[0].bar(labels, best_seq); ax[0].set_ylabel('best sequential RMSE (normalised angle)'); ax[0].tick_params(axis='x', rotation=30)
ax[1].scatter(results['MACs'] / 1e6, best_seq)
for l, x, y in zip(labels, results['MACs'] / 1e6, best_seq): ax[1].annotate(l, (x, y), fontsize=8)
ax[1].set_xlabel('MACs per inference (M)'); ax[1].set_ylabel('best sequential RMSE'); ax[1].set_xscale('log')
plt.tight_layout(); plt.savefig(f"{base['save_dir']}/ablation.png", dpi=120); plt.show()
print('Reference: the current HLS design does ~37 M MACs at ~1 MAC/cycle = ~374 ms at 100 MHz against a 25 ms stride.')